Загрузка данных в DataFrame

In [11]:
import pandas as pd

df = pd.read_csv("data/merged_dataset.csv")

df.info()

display(df.shape)

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7572 entries, 0 to 7571
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   review               7572 non-null   object 
 1   clean_review         5037 non-null   object 
 2   predicted_sentiment  7572 non-null   object 
 3   clinic               7572 non-null   int64  
 4   doctor               7572 non-null   int64  
 5   service              7572 non-null   int64  
 6   sentiment_score      6028 non-null   float64
dtypes: float64(1), int64(3), object(3)
memory usage: 414.2+ KB


(7572, 7)

,review,clean_review,predicted_sentiment,clinic,doctor,service,sentiment_score
0,"Обратились в поликлинику, специалист Сергей Ви...",обратиться поликлиника специалист сергей вик...,POSITIVE,1,0,0,0.981365
1,Обратился в эту клинику и был приятно удивлен ...,обратиться клиника приятно удивлённый качество...,POSITIVE,1,1,1,0.981429
2,Прийти на прием к проктологу - это ведь не к т...,прийти приём проктолог это терапевт насморк...,POSITIVE,0,1,0,0.981413
3,"От всей души благодарю весь коллектив ""Уро-Про...",весь душа благодарить весь коллектив уро ...,POSITIVE,0,1,0,0.981429
4,Хочу выразить благодарность доктору Можаевой Е...,хотеть выразить благодарность доктор можаевой ...,POSITIVE,0,1,0,0.981412


Получение сведений о пропущенных данных

Типы пропущенных данных:

None - представление пустых данных в Python
NaN - представление пустых данных в Pandas
'' - пустая строка

In [12]:
# Количество пустых значений признаков
display(df.isnull().sum())
display()

# Есть ли пустые значения признаков
display(df.isnull().any())
display()

# Процент пустых значений признаков
for i in df.columns:
    null_rate = df[i].isnull().sum() / len(df) * 100
    if null_rate > 0:
        display(f"{i} процент пустых значений: %{null_rate:.2f}")

review                    0
clean_review           2535
predicted_sentiment       0
clinic                    0
doctor                    0
service                   0
sentiment_score        1544
dtype: int64

review                 False
clean_review            True
predicted_sentiment    False
clinic                 False
doctor                 False
service                False
sentiment_score         True
dtype: bool

'clean_review процент пустых значений: %33.48'

'sentiment_score процент пустых значений: %20.39'

Заполнение пропущенных данных

In [13]:
fillna_df = df.fillna(0)

display(fillna_df.shape)

display(fillna_df.isnull().any())

# Замена пустых данных на 0
df["clean_review"] = df["clean_review"].fillna(0)

# Замена пустых данных на медиану
df["sentiment_score"] = df["sentiment_score"].fillna(df["sentiment_score"].median())

df.tail()

(7572, 7)

review                 False
clean_review           False
predicted_sentiment    False
clinic                 False
doctor                 False
service                False
sentiment_score        False
dtype: bool

,review,clean_review,predicted_sentiment,clinic,doctor,service,sentiment_score
7567,Издеваются над людьми! Держат в заложниках эмб...,0,NEUTRAL,0,0,0,0.981061
7568,Добрый день\nСегодня я обратилась в эту клиник...,0,NEUTRAL,0,0,0,0.981061
7569,"Говорят, основываясь на данных, в которых толь...",0,NEUTRAL,0,0,0,0.981061
7570,"Была на приёме ,врач не опытный ,не воспитанны...",0,NEUTRAL,0,1,0,0.981061
7571,Десна после установки коронки этим доктором по...,0,NEUTRAL,0,1,0,0.981061


Создание выборок данных

In [14]:
# Вывод распределения количества наблюдений по меткам (классам)
from src.utils import split_stratified_into_train_val_test

display(df.predicted_sentiment.value_counts())
display()

data = df[["predicted_sentiment", "sentiment_score", "doctor", "clinic", "service"]].copy()

df_train, df_val, df_test, y_train, y_val, y_test = split_stratified_into_train_val_test(
   data, stratify_colname="predicted_sentiment", frac_train=0.60, frac_val=0.20, frac_test=0.20
)

display("Обучающая выборка: ", df_train.shape)
display(df_train.predicted_sentiment.value_counts())

display("Контрольная выборка: ", df_val.shape)
display(df_val.predicted_sentiment.value_counts())

display("Тестовая выборка: ", df_test.shape)
display(df_test.predicted_sentiment.value_counts())

predicted_sentiment
POSITIVE    4137
NEGATIVE    2616
NEUTRAL      819
Name: count, dtype: int64

'Обучающая выборка: '

(4543, 5)

predicted_sentiment
POSITIVE    2482
NEGATIVE    1570
NEUTRAL      491
Name: count, dtype: int64

'Контрольная выборка: '

(1514, 5)

predicted_sentiment
POSITIVE    827
NEGATIVE    523
NEUTRAL     164
Name: count, dtype: int64

'Тестовая выборка: '

(1515, 5)

predicted_sentiment
POSITIVE    828
NEGATIVE    523
NEUTRAL     164
Name: count, dtype: int64

Выборка с избытком (oversampling)

In [15]:
from imblearn.over_sampling import SMOTE

# Убедимся, что целевая переменная закодирована числами
df_train["predicted_sentiment"] = df_train["predicted_sentiment"].map({"NEGATIVE": -1, "NEUTRAL": 0, "POSITIVE": 1})

smote = SMOTE()

display("Обучающая выборка: ", df_train.shape)
display(df_train.predicted_sentiment.value_counts())

X_resampled, y_resampled = smote.fit_resample(df_train, df_train["predicted_sentiment"]) # type: ignore
df_train_smote = pd.DataFrame(X_resampled)

display("Обучающая выборка после oversampling: ", df_train_smote.shape)
display(df_train_smote.predicted_sentiment.value_counts())

df_train_smote

'Обучающая выборка: '

(4543, 5)

predicted_sentiment
 1    2482
-1    1570
 0     491
Name: count, dtype: int64

/Users/a.makienko/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


'Обучающая выборка после oversampling: '

(7446, 5)

predicted_sentiment
 1    2482
-1    2482
 0    2482
Name: count, dtype: int64

,predicted_sentiment,sentiment_score,doctor,clinic,service
0,1,0.981341,1,1,0
1,-1,0.287314,0,0,0
2,0,0.981061,1,0,0
3,-1,0.316561,1,0,0
4,-1,0.981061,0,0,1
...,...,...,...,...,...
7441,0,0.981061,1,1,0
7442,0,0.812695,1,0,0
7443,0,0.810521,0,0,0
7444,0,0.981061,1,0,0


Выборка с недостатком (undersampling)

In [16]:
from imblearn.under_sampling import RandomUnderSampler

undersampler = RandomUnderSampler()

display("Обучающая выборка: ", df_train.shape)
display(df_train.predicted_sentiment.value_counts())

X_resampled, y_resampled = undersampler.fit_resample(df_train, df_train["predicted_sentiment"])  # type: ignore
df_train_under = pd.DataFrame(X_resampled)

display("Обучающая выборка после undersampling: ", df_train_under.shape)
display(df_train_under.predicted_sentiment.value_counts())

df_train_under

'Обучающая выборка: '

(4543, 5)

predicted_sentiment
 1    2482
-1    1570
 0     491
Name: count, dtype: int64

/Users/a.makienko/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/a.makienko/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


'Обучающая выборка после undersampling: '

(1473, 5)

predicted_sentiment
-1    491
 0    491
 1    491
Name: count, dtype: int64

,predicted_sentiment,sentiment_score,doctor,clinic,service
5229,-1,0.388243,1,0,0
688,-1,0.751432,1,1,0
6442,-1,0.981061,0,1,0
725,-1,0.751445,0,0,0
6420,-1,0.981061,1,1,0
...,...,...,...,...,...
3144,1,0.980849,1,1,1
132,1,0.981390,1,0,0
3396,1,0.980887,1,0,1
1944,1,0.981311,1,0,0
